In [11]:
import xarray as xr
import os
import pandas as pd

Juntando todos os arquivo .nc do conjunto de dados era5 e era5 land


In [12]:


# Caminho para a pasta onde estão os arquivos .nc
pasta = "/home/lacrio2/Documentos/LACRIO/PROJETO/LACRIO/Analises/ERA5_dados/dados_era5_land"
arquivos_nc = sorted([os.path.join(pasta, f) for f in os.listdir(pasta) if f.endswith(".nc")])

# Abrir e concatenar ao longo da dimensão "time" (ou outra, se necessário)
ds_era5_land = xr.open_mfdataset(arquivos_nc, combine='by_coords')

# Caminho para a pasta onde estão os arquivos .nc
pasta = "/home/lacrio2/Documentos/LACRIO/PROJETO/LACRIO/Analises/ERA5_dados/dados_era5"
arquivos_nc = sorted([os.path.join(pasta, f) for f in os.listdir(pasta) if f.endswith(".nc")])

# Abrir e concatenar ao longo da dimensão "time" (ou outra, se necessário)
ds_era5 = xr.open_mfdataset(arquivos_nc, combine='by_coords')


Criando um arquivo csv para a localização da estação Cuchillacocha para o arquivo era5 land

In [13]:


# Coordenadas da estação
lat_estacao = -9.41
lon_estacao = -77.35

# Abrir o arquivo NetCDF
ds = ds_era5_land

# Verifique os nomes corretos das dimensões
print(ds.dims)
print(ds.coords)

# Substituir nomes de coordenadas se necessário
if 'lat' in ds.coords and 'lon' in ds.coords:
    ds = ds.rename({'lat': 'latitude', 'lon': 'longitude'})

# Encontrar os índices mais próximos da coordenada desejada
lat_idx = abs(ds['latitude'] - lat_estacao).argmin()
lon_idx = abs(ds['longitude'] - lon_estacao).argmin()

# Selecionar os dados no ponto mais próximo
ponto = ds.isel(latitude=lat_idx, longitude=lon_idx)

# Usar 'valid_time' como eixo temporal
tempo = ds['valid_time'].values

# Extrair dados para todas as variáveis dependentes de 'valid_time'
dados = {}
for var in ds.data_vars:
    dims = ds[var].dims
    if 'valid_time' in dims:
        dados[var] = ponto[var].values
    else:
        # Repete o valor único ao longo do tempo
        dados[var] = [ponto[var].values] * len(tempo)

# Criar DataFrame
df = pd.DataFrame(dados)
df["valid_time"] = tempo
ds_era5_land_csv = df[["valid_time"] + [v for v in dados if v != "valid_time"]]



FrozenMappingWarningOnValuesAccess({'valid_time': 118344, 'latitude': 6, 'longitude': 4})
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 947kB 1940-01-01 ... 2020-12-31T1...
  * latitude    (latitude) float64 48B -8.65 -8.9 -9.15 -9.4 -9.65 -9.9
  * longitude   (longitude) float64 32B -77.9 -77.65 -77.4 -77.15
    number      int64 8B 0
    expver      (valid_time) object 947kB dask.array<chunksize=(1464,), meta=np.ndarray>


Criando um arquivo csv para a localização da estação Cuchillacocha para o arquivo era5 

In [14]:
import xarray as xr
import pandas as pd

# Coordenadas do ponto de interesse (Cuchillacocha)
lat_estacao = -9.41
lon_estacao = -77.35

# Abrir dataset
ds = ds_era5

# Renomear lat/lon se necessário
if 'lat' in ds.coords or 'lon' in ds.coords:
    ds = ds.rename({'lat': 'latitude', 'lon': 'longitude'})

# Achar índice do ponto mais próximo
lat_idx = abs(ds['latitude'] - lat_estacao).argmin()
lon_idx = abs(ds['longitude'] - lon_estacao).argmin()

# Verifica se existe a dimensão pressure_level
if "pressure_level" not in ds.dims:
    raise ValueError("O dataset não possui a dimensão 'pressure_level'.")

# Criar dicionário com DataFrames por nível de pressão
dfs_por_pressao = {}

for nivel in ds.pressure_level.values:
    ponto_nivel = ds.isel(latitude=lat_idx, longitude=lon_idx).sel(pressure_level=nivel)
    tempo = ds["valid_time"].values
    dados = {}

    for var in ds.data_vars:
        da = ponto_nivel[var]
        if "valid_time" in da.dims:
            # Reduz outras dimensões, se houver
            dims_extras = [d for d in da.dims if d != "valid_time"]
            if dims_extras:
                da = da.isel({d: 0 for d in dims_extras})
            dados[var] = da.values
        else:
            dados[var] = [da.values.item()] * len(tempo)

    df = pd.DataFrame(dados)
    df["valid_time"] = tempo
    df = df[["valid_time"] + [v for v in dados if v != "valid_time"]]

    dfs_por_pressao[int(nivel)] = df

# Empilhar todos os DataFrames com coluna 'pressure_level'
lista_df = []

for nivel, df in dfs_por_pressao.items():
    df_com_nivel = df.copy()
    df_com_nivel["pressure_level"] = nivel
    lista_df.append(df_com_nivel)

df_empilhado = pd.concat(lista_df, ignore_index=True)

# Reorganizar colunas (opcional)
colunas_ordenadas = ["valid_time", "pressure_level"] + [col for col in df_empilhado.columns if col not in ["valid_time", "pressure_level"]]
df_era5_csv = df_empilhado[colunas_ordenadas]

# ✅ df_empilhado agora contém todos os dados organizados por tempo e nível de pressão
print("✅ DataFrame final criado com sucesso!")

# Exemplo de uso:
# print(df_empilhado.head())


✅ DataFrame final criado com sucesso!


In [15]:
df_era5_csv

,valid_time,pressure_level,z,q,crwc,t,u,v
0,1940-01-01 00:00:00,500,57489.031250,0.002608,0.0,268.483124,0.371095,6.677262
1,1940-01-01 01:00:00,500,57528.335938,0.002191,0.0,268.860870,0.057231,6.478654
2,1940-01-01 02:00:00,500,57564.855469,0.001929,0.0,269.163086,-0.361216,6.226668
3,1940-01-01 03:00:00,500,57622.351562,0.001793,0.0,269.276062,-1.235389,5.597896
4,1940-01-01 04:00:00,500,57657.062500,0.001728,0.0,269.255127,-1.945148,4.694727
...,...,...,...,...,...,...,...,...
2125795,2020-11-15 19:00:00,300,95100.453125,0.000348,0.0,241.747437,-0.385240,1.356631
2125796,2020-11-15 20:00:00,300,95067.328125,0.000429,0.0,241.928040,-1.312189,1.560719
2125797,2020-11-15 21:00:00,300,95036.671875,0.000498,0.0,242.342667,-1.452204,2.107743
2125798,2020-11-15 22:00:00,300,95046.625000,0.000434,0.0,242.087677,-1.116784,4.098901


In [16]:
ds_era5_land_csv

,valid_time,tp,ssrd,strd,sf,u10,v10,d2m,t2m,sp,z
0,1940-01-01 00:00:00,NaN,NaN,NaN,NaN,0.783538,0.297100,275.532715,276.785156,61884.890625,40786.863281
1,1940-01-01 06:00:00,NaN,NaN,NaN,NaN,0.341216,0.619019,272.547119,272.978027,61864.550781,40786.863281
2,1940-01-01 12:00:00,0.000000,2.376755e+05,9.551199e+05,0.000000,0.150949,0.803593,272.044434,273.303223,61907.886719,40786.863281
3,1940-01-01 18:00:00,0.000029,3.040179e+06,1.019646e+06,0.000000,0.506329,-0.373855,274.152344,285.407471,61900.890625,40786.863281
4,1940-01-02 00:00:00,0.000098,3.316509e+04,1.015704e+06,0.000000,1.050761,0.431399,275.730713,278.266113,61821.378906,40786.863281
...,...,...,...,...,...,...,...,...,...,...,...
118339,2020-12-30 18:00:00,0.000172,3.049215e+06,1.056858e+06,0.000000,1.051177,-0.253581,273.410400,283.217773,61820.363281,40786.863281
118340,2020-12-31 00:00:00,0.000071,2.795501e+04,1.078665e+06,0.000000,0.390170,0.274296,276.025391,279.043701,61774.507812,40786.863281
118341,2020-12-31 06:00:00,0.000007,0.000000e+00,9.992986e+05,0.000000,0.559186,0.480747,274.952881,275.807373,61783.039062,40786.863281
118342,2020-12-31 12:00:00,0.000185,2.211733e+05,1.078082e+06,0.000062,-0.012476,0.311699,275.026611,277.200684,61835.253906,40786.863281
